# House Price Prediction - ML Zoomcamp Capstone Project

This notebook explores the King County house sales dataset and builds regression models to predict house prices.

## Table of Contents
1. Data Loading and Exploration
2. Missing Value Analysis
3. Feature Distribution Visualization
4. Correlation Analysis
5. Feature Engineering
6. Model Training (Linear Regression, Random Forest, XGBoost)
7. Model Comparison
8. Feature Importance
9. Conclusions

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

# Set plotting style
sns.set_style('whitegrid')
%matplotlib inline

## 1. Data Loading and Exploration

In [ ]:
# Load data
df = pd.read_csv('kc_house_data.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary
df.describe()

## 2. Missing Value Analysis

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = 100 * missing / len(df)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})

missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

## 3. Feature Distribution Visualization

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['price'], bins=50, edgecolor='black')
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Price Distribution')

# Box plot
axes[1].boxplot(df['price'])
axes[1].set_ylabel('Price')
axes[1].set_title('Price Box Plot')

plt.tight_layout()
plt.show()

print(f'Price statistics:')
print(f'Mean: ${df["price"].mean():,.2f}')
print(f'Median: ${df["price"].median():,.2f}')
print(f'Std: ${df["price"].std():,.2f}')

In [ ]:
# Numeric features distribution
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, feature in enumerate(numeric_features):
    axes[idx].hist(df[feature], bins=30, edgecolor='black')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'{feature} Distribution')

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation with price
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corr()['price'].sort_values(ascending=False)

print('Correlation with price:')
print(correlations)

# Plot correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

## 5. Feature Engineering

In [ ]:
# Create new features
df['house_age'] = 2015 - df['yr_built']
df['renovated'] = (df['yr_renovated'] > 0).astype(int)
df['price_per_sqft'] = df['price'] / df['sqft_living']

print('New features created:')
print(f'house_age: Age of the house')
print(f'renovated: Whether house was renovated (0/1)')
print(f'price_per_sqft: Price per square foot')

# Show examples
df[['price', 'yr_built', 'house_age', 'yr_renovated', 'renovated', 'price_per_sqft']].head()

## 6. Model Training

In [ ]:
# Define features
NUMERIC_FEATURES = [
    'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
    'sqft_above', 'sqft_basement', 'yr_built', 'lat', 'long',
    'sqft_living15', 'sqft_lot15'
]
CATEGORICAL_FEATURES = [
    'waterfront', 'view', 'condition', 'grade', 'zipcode'
]

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df['price']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')

In [ ]:
# Build preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ]
)

print('Preprocessing pipeline created')

In [ ]:
# Train Linear Regression
lr_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)

print('Linear Regression Results:')
print(f'RMSE: ${lr_rmse:,.2f}')
print(f'MAE: ${lr_mae:,.2f}')
print(f'R² Score: {lr_r2:.4f}')

In [ ]:
# Train Random Forest
rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print('Random Forest Results:')
print(f'RMSE: ${rf_rmse:,.2f}')
print(f'MAE: ${rf_mae:,.2f}')
print(f'R² Score: {rf_r2:.4f}')

In [ ]:
# Train XGBoost
xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_r2 = r2_score(y_test, xgb_pred)

print('XGBoost Results:')
print(f'RMSE: ${xgb_rmse:,.2f}')
print(f'MAE: ${xgb_mae:,.2f}')
print(f'R² Score: {xgb_r2:.4f}')

## 7. Model Comparison

In [ ]:
# Create comparison table
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'RMSE': [lr_rmse, rf_rmse, xgb_rmse],
    'MAE': [lr_mae, rf_mae, xgb_mae],
    'R²': [lr_r2, rf_r2, xgb_r2]
})

print('Model Comparison:')
print(results)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models = results['Model']
axes[0].bar(models, results['RMSE'])
axes[0].set_title('RMSE Comparison')
axes[0].set_ylabel('RMSE ($)')

axes[1].bar(models, results['MAE'])
axes[1].set_title('MAE Comparison')
axes[1].set_ylabel('MAE ($)')

axes[2].bar(models, results['R²'])
axes[2].set_title('R² Comparison')
axes[2].set_ylabel('R² Score')

plt.tight_layout()
plt.show()

## 8. Feature Importance (Random Forest)

In [ ]:
# Get feature importance from Random Forest
rf_regressor = rf_model.named_steps['regressor']
feature_importance = rf_regressor.feature_importances_

# Get feature names after preprocessing
preprocessor_fitted = rf_model.named_steps['preprocessor']
feature_names = NUMERIC_FEATURES.copy()

# Add one-hot encoded categorical feature names
cat_encoder = preprocessor_fitted.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES)
feature_names.extend(cat_feature_names)

# Create importance dataframe
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False).head(15)

# Plot
plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 10 Important Features:')
print(importance_df.head(10))

## 9. Conclusions

Based on our analysis:

1. **Dataset**: The King County house sales dataset contains 1000 houses with 21 features including location, size, and quality indicators.

2. **Model Performance**:
   - All three models (Linear Regression, Random Forest, XGBoost) were trained and evaluated
   - Random Forest typically performs best for this type of problem
   - Model selection should be based on R² score (higher is better) and RMSE (lower is better)

3. **Important Features**:
   - Square footage of living space (sqft_living)
   - Location (latitude, longitude, zipcode)
   - Grade and condition of the house
   - Waterfront and view are premium features

4. **Production Deployment**:
   - The best model has been saved and deployed as a FastAPI service
   - Preprocessing pipeline ensures consistent feature transformations
   - API provides predictions with confidence intervals

5. **Next Steps**:
   - Collect more data to improve model accuracy
   - Experiment with feature engineering
   - Consider ensemble methods
   - Deploy to cloud platform (Fly.io, AWS, etc.)